# Approach 3: BiLSTM / BiGRU on Colab
Runs the shared preprocessing pipeline and the BiLSTM/BiGRU experiments for the GBV tweet classification project.

Use **Runtime > Change runtime type > GPU**. Two modes:
- `RUN_FULL = False` (default): skips training and rebuilds all tables/figures from the results committed in `results/bilstm_gru/runs/`. These files must be in the repo; the notebook stops with a clear message if they are missing.
- `RUN_FULL = True`: retrains all 17 runs (9 design-choice runs at seed 42 + seeds 1 and 2 for the 4 final models; resumable: finished runs are skipped). Slow, so keep the runtime connected.

In [ ]:
TEAM_REPO_URL = "https://github.com/eldrige/nlp_language_tech_formative.git"
BRANCH = "bilstm-gru"     # change if your code is on another branch
REPO_DIR = "/content/repo"
RUN_FULL = False

## 1. Clone the repo and install what Colab lacks

In [ ]:
import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, TEAM_REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
%cd {REPO_DIR}
!ls
assert os.path.exists("src/preprocessing.py"), (
    "src/preprocessing.py not found: this repo/branch does not contain the pipeline. "
    "Check TEAM_REPO_URL and BRANCH in the first cell.")
# Colab already has TensorFlow/Keras 3; only make sure the rest is present
!pip install -q scikit-learn pandas matplotlib

## 2. Data
The Zindi CSVs are in the repo root. `preprocessing.py` finds them there (or in `data/raw/`). If they are missing, upload them.

In [ ]:
import os
need = ["Train.csv", "Test.csv", "SampleSubmission.csv"]
have = lambda f: os.path.exists(f) or os.path.exists("data/raw/" + f)
if not all(have(f) for f in need):
    from google.colab import files
    for name in files.upload():
        os.makedirs("data/raw", exist_ok=True); os.replace(name, "data/raw/" + name)
print({f: have(f) for f in need})

## 3. Pretrained GloVe-Twitter embeddings (~390 MB)

In [ ]:
os.makedirs("embeddings", exist_ok=True)
if not os.path.exists("embeddings/glove-twitter-100.gz"):
    !curl -L -o embeddings/glove-twitter-100.gz https://github.com/RaRe-Technologies/gensim-data/releases/download/glove-twitter-100/glove-twitter-100.gz
!ls -lh embeddings

## 4. Check the shared pipeline

In [ ]:
import sys; sys.path.append("src")
from preprocessing import load_data, prepare_sequences, LABELS
train, val, test, zindi = load_data()
print("train/val/test/zindi:", len(train), len(val), len(test), len(zindi))   # expect 31223 3904 3903 15581
d = prepare_sequences(max_len=80)
print("X_train:", d["X_train"].shape, "| vocab size:", len(d["vocab"]))         # expect (31223, 80) 16634

## 5. Train (optional), then build tables and figures

In [ ]:
%cd {REPO_DIR}/src
if RUN_FULL:
    !python bilstm_experiments.py
else:
    print("RUN_FULL=False: using committed results")

In [ ]:
runs = f"{REPO_DIR}/results/bilstm_gru/runs"
if not RUN_FULL and not (os.path.isdir(runs) and os.listdir(runs)):
    raise FileNotFoundError(
        "No saved runs in results/bilstm_gru/runs. Commit them to the repo, "
        "or set RUN_FULL = True in the first cell to retrain.")
!python bilstm_report.py

## 6. Results

In [ ]:
import glob, pandas as pd
from IPython.display import Image, display
os.chdir(REPO_DIR)
for f in ["summary_seeds.csv", "summary_runs.csv"]:
    p = "results/bilstm_gru/" + f
    if os.path.exists(p):
        print(f); display(pd.read_csv(p))
for f in sorted(glob.glob("results/bilstm_gru/fig_*.png")):
    print(f); display(Image(f))

## 7. Check the shared prediction files

In [ ]:
EXPECTED = ["Tweet_ID","tweet","true","pred","prob_sexual_violence","prob_Physical_violence",
            "prob_emotional_violence","prob_economic_violence","prob_Harmful_Traditional_practice"]
files_ = glob.glob("results/predictions/*.csv")
print(files_)
for p in files_:
    df = pd.read_csv(p)
    print(p, df.shape, "columns OK" if list(df.columns) == EXPECTED else list(df.columns))